In [1]:
from Transformer_module.transformer_engine import TransformerDetector
from rule_filter import EmailBlacklistFilter
from CNN_module.cnn_engine import CNNDetector
from SNN_module.snn_engine import SNNDetector
from email_scanner import EmailAddressScanner
from deep_translator import GoogleTranslator

blacklist_guard = EmailBlacklistFilter()
cnn_checker = CNNDetector()
snn_checker = SNNDetector()
transformer_checker = TransformerDetector()
address_checker = EmailAddressScanner()
translator = GoogleTranslator(source_lang="auto", target_lang="en")


def classify_incoming_mail(sender, content, verbose=True):
    is_spam, reason = blacklist_guard.check_spam(sender)

    if is_spam:
        print(f"PHÁN QUYẾT: THƯ RÁC (Spam) - Lý do: {reason}")

    if verbose:
        address_result = address_checker.scan(sender)
        if "error" in address_result:
            print("-" * 50)
            print(f"TỪ CHỐI NHẬN: {address_result['error']}")
            return "ERROR_FORMAT"
        
        if address_result['is_spam'] == True:
            blacklist_guard.add_to_blacklist(sender)
            print(f"Kết quả kiểm tra địa chỉ email: {address_result['status']}")
            print(f"Xác suất Spam: {address_result['spam_prob']}%")
        else:
            try:
                content_en = translator.translate(content)
            except Exception as e:
                content_en = content
                
            print(f"Dự đoán với CNN_model:")
            cnn_prob = cnn_checker.predict(content_en)
            print(f"Thư rác" if cnn_prob > 50 else f"Thư thường")
            print(f"Độ tin cậy: {max(cnn_prob, 100 - cnn_prob):.2f}")
            print(f"Dự đoán với Transformer_model:")
            transformer_prob = transformer_checker.predict(content_en)
            print(f"Thư rác" if transformer_prob > 50 else f"Thư thường")
            print(f"Độ tin cậy: {max(transformer_prob, 100 - transformer_prob):.2f}")
            print(f"Dự đoán với SNN_model:")
            snn_prob = snn_checker.predict(content_en)
            print(f"Thư rác" if snn_prob > 50 else f"Thư thường")
            print(f"Độ tin cậy: {max(snn_prob, 100 - snn_prob):.2f}")
            print("-" * 50)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: c:\Users\ADMIN\Downloads\Code\.vscode\SPAM_CLASSIFIER\Transformer_module\my_bert_model
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
classify_incoming_mail("test@example.com", "this is a test email content to check if the spam filter works correctly. It should not be classified as spam.")

Dự đoán với CNN_model:
Thư thường
Độ tin cậy: 99.85
Dự đoán với Transformer_model:
Thư thường
Độ tin cậy: 99.97
Dự đoán với SNN_model:
Thư thường
Độ tin cậy: 98.20
--------------------------------------------------
